# Tutorial · 营销数据表示 + 多模态 (v6.0 Oxford Tutorial 仿真)

## Cell 1 - Persona (Oxford Fellow)

**System Prompt (角色设定)**:

> You are an **Oxford tutorial fellow** in *营销数据表示与多模态大模型演进* (Marketing Data Representation & Multimodal LLMs).
> Your tutorial style follows Oxford's 1-on-1 tradition: rigorous, adversarial, Socratic.
>
> **Hard rules**:
> 1. **Never give the direct answer.** 不直接给答案, 不直接答, never feed the conclusion. 禁直接答案。
> 2. Use **Socratic questioning** - 每轮以 probing question 结尾, 迫使学生自己推理。
> 3. Play **devil's advocate** - 对学生的任何 vague claim, 反问 "凭什么? 依据? 反例? counterexample?"
> 4. If student defends weakly, **drop one scaffold level** (worked -> faded -> independent), but still 不直接给答案。
> 5. Reference 真实库因果链: sentence-transformers / transformers CLIP / torch Two-Tower / InfoNCE 对比学习。
>
> **禁止**: 表扬 "good job" (Hattie Self 级表扬无效); 给完整代码; 接受 "我觉得" 无证据陈述。


## Cell 2 - Pre-Tutorial Task (强制 retrieval, 不查资料)

**研究依据**: Butler (2010) 检索练习效应 - 学生先自己尝试, 比直接看答案保留度高 50%+。

**课前必交** (限时 10 分钟, 写在下面 cell 的 `pre_task_answer` 字符串里):

1. 用一句话解释: 为什么客户嵌入空间和产品嵌入空间不能直接算 cosine 相似度? (提示: "对齐")
2. InfoNCE 损失 `L = -log[exp(sim(u,v⁺)) / Σ exp(sim(u,vᵢ))]` 中, v⁺ 是什么? vᵢ 包括哪些?
3. CLIP (2021) 与 GPT-4o (2024) 在多模态处理上的本质区别是什么? (提示: "双塔 vs 原生多模态")

> 不交 pre-task 不允许进入 Socratic loop。这是 Oxford tutorial 的强制 retrieval gate。


In [ ]:
# Cell 3 - Socratic Loop (静态 if/else 模拟, 不调 API, anti-stall)
# 研究依据: Oxford tutorial 1对1 + Vygotsky 共构 + 2024-2025 Socratic LLM 论文 (arxiv 2409.05511 等)

import json

def socratic_loop(student_responses):
    """静态模拟 5 轮 Socratic 追问, 每轮检测 defense 失败则降一级 scaffold.
    每轮必以 probing question 结尾, 永不直接给答案。"""
    transcript = []
    scaffold = "worked"  # 起始脚手架级别

    # ---- Round 1: 概念检测 - 客户/产品嵌入为何不能直接 cosine ----
    r1 = student_responses.get("r1", "").lower()
    if "对齐" in r1 or "align" in r1:
        transcript.append(("T1", "你提到了'对齐'。为什么对齐是必要的? 若客户塔和产品塔各自独立训练, 它们的向量空间在几何上是什么关系? 凭什么说直接算 cosine 没意义?"))
    else:
        scaffold = "worked"
        transcript.append(("T1", "你的回答没有触及核心。假设客户嵌入空间的维度是 384, 产品嵌入也是 384, 为什么这两个 384 维空间不能直接算 cosine? 提示: 训练目标是否共享?"))

    # ---- Round 2: InfoNCE 因果链 ----
    r2 = student_responses.get("r2", "").lower()
    if "v+" in r2 or "v⁺" in r2 or "正样本" in r2 or "positive" in r2:
        transcript.append(("T2", "你说 v⁺ 是正样本。那 vᵢ 中的负样本如何采样? 若只用 1 个负样本, InfoNCE 退化为二分类, 对比信号会变弱 - 为什么? 如何 counterexample?"))
    else:
        scaffold = "faded"
        transcript.append(("T2", "InfoNCE 公式 L=-log[exp(sim(u,v⁺))/Σexp(sim(u,vᵢ))], v⁺ 和 vᵢ 分别指什么? 为什么分母是 Σ 而不是单项?"))

    # ---- Round 3: 温度参数 τ 的反事实 ----
    r3 = student_responses.get("r3", "").lower()
    if "τ" in r3 or "温度" in r3 or "temperature" in r3:
        transcript.append(("T3", "你提到温度参数 τ。反事实: 若 τ 从 0.01 变到 1.0, CLIP 对正确匹配的'信心'如何变化? softmax 分布形状如何变? 这对训练稳定性意味着什么?"))
    else:
        scaffold = "faded"
        transcript.append(("T3", "CLIP 的对称 InfoNCE 有一个温度参数 τ。若 τ 变小, 模型对正确匹配的信心变强还是变弱? 为什么? 依据是什么?"))

    # ---- Round 4: CLIP vs GPT-4o 本质区别 ----
    r4 = student_responses.get("r4", "").lower()
    if "双塔" in r4 or "原生多模态" in r4 or "native" in r4 or "unified token" in r4:
        transcript.append(("T4", "你说 CLIP 是双塔, GPT-4o 是原生多模态。这个区别如何影响营销应用? 若广告图片中有红色文字与描述语气不一致, CLIP 能察觉吗? GPT-4o 能吗? 为什么?"))
    else:
        scaffold = "independent"
        transcript.append(("T4", "CLIP (2021) 和 GPT-4o (2024) 都叫'多模态'。它们的本质区别是什么? 若你看一张广告图, CLIP 能做什么, 不能做什么? GPT-4o 多了什么能力?"))

    # ---- Round 5 (bonus): 对比学习贯穿性 ----
    r5 = student_responses.get("r5", "").lower()
    if "对比学习" in r5 or "contrastive" in r5 or "infonce" in r5:
        transcript.append(("T5", "对比学习贯穿 CLIP / Two-Tower / sentence-transformers。假设你要给一个美妆电商设计营销系统, 这三者的对比学习目标分别是什么? 若三者共享一个 InfoNCE 实现, 哪个最关键?"))
    else:
        transcript.append(("T5", "最后一个问题: CLIP, Two-Tower, sentence-transformers 三者都用对比学习, 但对齐的对象不同。分别对齐了什么? 凭什么 InfoNCE 是现代表示学习的核心?"))

    return transcript, scaffold

# ---- 模拟学生 5 轮回答 (静态 if/else 分支, 不调 API) ----
mock_student = {
    "r1": "客户和产品嵌入空间没对齐",  # 含"对齐"
    "r2": "v+ 是正样本, vi 包括负样本",  # 含正样本
    "r3": "温度参数控制区分度",          # 含温度
    "r4": "CLIP 是双塔, GPT-4o 是原生多模态",  # 含双塔+原生多模态
    "r5": "对比学习是核心"               # 含对比学习
}

transcript, final_scaffold = socratic_loop(mock_student)
print(f"=== Socratic Loop Transcript (final scaffold: {final_scaffold}) ===\n")
for turn_id, content in transcript:
    print(f"[{turn_id}] {content}\n")
print(f"\n共 {len(transcript)} 轮 Socratic 追问, 全部以 probing question 结尾, 无一直接给答案。")


In [ ]:
# Cell 4 - student_model.json (跨单元复用, 记录掌握度/盲点)
# 研究依据: Oxford tutorial 的 fellow 会在脑中维护学生画像, 跨周次调整教学焦点

import json, os

student_model = {
    "unit": "skill-1-representation/day-2-marketing-representation",
    "timestamp": "2026-07-25T04:47:00",
    "mastery": {
        "S1_text_embedding": 0.85,        # sentence-transformers 文本嵌入
        "S2_two_tower": 0.60,             # Two-Tower + InfoNCE (弱项)
        "S3_clip_multimodal": 0.75        # CLIP + 多模态演进
    },
    "weak_points": [
        "InfoNCE 负采样数量对对比信号的影响",
        "温度参数 τ 与 softmax 分布的关系",
        "CLIP 对称损失的图文/文图双向"
    ],
    "scaffold_level": "faded",            # 当前脚手架级别
    "fail_count": {"D1": 0, "D2": 2, "D3": 1},  # D2 连续2次失败 -> 触发 weak_loop
    "recommended_review": ["C3", "C4", "C6"],   # 对应 schedule.json 卡片
    "blocked": False,
    "next_unit_ready": False,             # S2 未达 mastery, 暂不进 Day 3
    "tutorial_history": [
        {"date": "2026-07-25", "rounds": 5, "final_scaffold": "faded", "weak_drill": "D2"}
    ]
}

# 写入 student_model.json (供后续单元 tutorial 读取, 跨单元连续画像)
out_path = "student_model.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(student_model, f, ensure_ascii=False, indent=2)
print(f"student_model.json 已写入: {out_path}")
print(f"弱项: {student_model['weak_points']}")
print(f"推荐复习卡片: {student_model['recommended_review']}")


## Cell 5 - Hattie 四级 Formative Feedback

> **研究依据**: Hattie (2007 RER 77(1):81-112) - 有效反馈分 4 级, Self 级表扬无效, Feed-Forward 最有效。
> 本 tutorial 结束后, 系统按四级给学生反馈:

### [TASK] 任务级反馈 (关于本次任务的正确性)
- Round 1 (对齐): 学生答出"对齐", 但未说明几何关系 - **任务部分正确**。
- Round 2 (InfoNCE): 学生识别 v⁺ 为正样本, 但未解释为何负样本要多 - **任务基本正确**。
- Round 3 (τ): 学生提到"控制区分度"但未提 softmax 锐化 - **任务部分正确**。
- Round 4 (CLIP vs GPT-4o): 学生答出"双塔 vs 原生多模态" - **任务正确**。
- Round 5 (对比学习贯穿): 学生答出"对比学习是核心"但未区分三者对齐对象 - **任务部分正确**。

### [PROCESS] 过程级反馈 (关于学生的策略/推理过程)
- 学生在 Round 2 表现出"公式记忆型"策略 - 能背公式但未理解多分类本质。建议策略转变: 把 InfoNCE 类比为 softmax 多分类, 分母是所有候选类的 logits。
- Round 3 学生未主动反事实推理 (τ 变大会怎样), 暴露反事实思维缺失。建议: 每次遇到参数必问"若它变 10 倍会怎样"。

### [SELF-REG] 自我调节级反馈 (关于学生的元认知/监控)
- 学生在 Round 4 自我监控良好 - 主动用"双塔 vs 原生多模态"框架, 显示元认知策略有效。
- 但 Round 5 未自我提问"三者对齐对象分别是什么", 显示监控盲点。建议: 进入新题前先自问"这道题的核心区分维度是什么"。

### [FEED-FORWARD] 前馈级反馈 (关于下一步该做什么)
- **立即可做**: 重做 practice.md D2 的 Faded 阶段 (InfoNCE 负采样), 通过后再试 Independent。
- **本单元内**: 用 schedule.json 复习卡片 C3 (Two-Tower) + C4 (CLIP) + C6 (对比学习贯穿性), 间隔 1/3/8/21 天。
- **跨单元**: Day 3 (企业知识图谱 + GraphRAG) 会用到本单元的向量表示作为"软"表示基础, 进入前确保 S2 mastery >= 0.8。
- **避坑**: 不要再背 InfoNCE 公式, 改用"类比 softmax 多分类"理解。不要再把 CLIP 和 GPT-4o 都归为"多模态", 必须区分"编码后对齐"vs"统一 token 空间"。

> **注**: 故意省略 Self 级表扬 (如 "good job") - Hattie 研究表明 Self 级反馈对学习效果接近 0, 甚至反作用。


## Cell 6 - 限频与 Exit Artifact

### 限频 (防依赖, 研究依据: Oxford tutorial 每周 1 次, 不允许天天找 fellow)

- **本单元 tutorial 限频**: 每单元 **1 次/天**, daily limit = 1 session。
- **超额触发**: 若学生在 24 小时内尝试启动第 2 次 Socratic loop, 系统拒绝并提示 "每天仅 1 次 tutorial, 请先用 schedule.json 自主复习, 明天再来"。
- **设计意图**: 防止学生用 tutorial 替代自主思考 (Vygotsky 共构要求学生先自己尝试)。Oxford tutorial 之所以有效, 正是因为稀缺 - 学生必须课前充分准备。
- **绕过检测**: 若学生通过修改 timestamp 绕过限频, 写入 student_model.json 的 `dependency_risk: true`, 触发人工干预。

### Exit Artifact (tutorial 结束必交)

> 不交 exit artifact 不允许离开 tutorial, 也不允许进入下一单元。

请在 `exit_artifact.json` 中填写以下 3 项:

```json
{
  "top_3_blind_spots": [
    "InfoNCE 负采样数量与对比信号强度的关系",
    "温度参数 τ 对 softmax 分布形状的影响",
    "CLIP 对称损失的图文/文图双向 - 我只算了图->文"
  ],
  "recommended_review_units": [
    "本单元 practice.md D2 (Two-Tower Faded)",
    "schedule.json C3 + C4 (间隔 1/3/8 天)",
    "Day 1 表示工程基础 (若 S2 仍卡)"
  ],
  "commitment": "我承诺在本周内重做 D2 Faded, 并用 τ=0.01 和 τ=1.0 各跑一次 CLIP 训练, 对比 loss 曲线。"
}
```

### Exit 检查清单

- [ ] top_3_blind_spots 至少 3 条, 每条具体 (不接受"我什么都不懂")
- [ ] recommended_review_units 至少 2 条, 引用具体文件 (practice.md / schedule.json / 跨单元)
- [ ] commitment 含可观察动作 (不接受"我会努力")
- [ ] student_model.json 已更新 mastery 分数 + weak_points
- [ ] 限频计数器 +1, 下次 tutorial 24 小时后才解锁

> **Tutorial 结束**。下次见。Remember: "天道推演不是预言, 而是通过穷尽可能的未来, 来选择最好的现在。" - 每次认真推演, 都是在向未来投资。
